# Week 1 — PTB-XL Data Audit

**Project:** Quality-Aware Synthetic Data Augmentation for Robust Imbalanced ECG Classification

**Goal (Week 1 only):** Understand the PTB-XL dataset, construct clean rhythm labels, perform EDA, document preprocessing decisions, create a patient-level split, and establish a simple baseline.

> We are **not** generating synthetic data, implementing quality metrics, or building the final CNN-LSTM in this notebook.

## 1. Project Setup

In [ ]:
import ast
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 11})

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# --- Paths (inspect project directory first) ---
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "archive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"

METADATA_PATH = DATA_DIR / "ptbxl_database.csv"
SCP_STATEMENTS_PATH = DATA_DIR / "scp_statements.csv"
RECORDS_DIR = DATA_DIR  # wfdb paths are relative to this root

SAMPLING_RATE = 100  # Hz (low-resolution records)
TARGET_RHYTHMS = ["SR", "AFIB", "SBRAD", "STACH"]

print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Metadata exists:", METADATA_PATH.exists())
print("SCP statements exist:", SCP_STATEMENTS_PATH.exists())
print("records100 exists:", (DATA_DIR / "records100").exists())

## 2. Load PTB-XL Metadata

In [ ]:
metadata = pd.read_csv(METADATA_PATH)
metadata.head()

## 3. Inspect Dataset Structure

In [ ]:
print(f"Total recordings: {len(metadata):,}")
print(f"Unique patients:  {metadata['patient_id'].nunique():,}")
print(f"Columns ({len(metadata.columns)}):\n", list(metadata.columns))
print("\nMissing values per column:")
print(metadata.isnull().sum()[metadata.isnull().sum() > 0])
print("\nRecordings per patient (summary):")
print(metadata.groupby("patient_id").size().describe())

## 4. Understand SCP Codes / Diagnostic Labels

PTB-XL stores annotations in the `scp_codes` column as a **stringified dictionary**.
Each key is an SCP statement code; each value is a confidence score (0–100).

The file `scp_statements.csv` maps each code to metadata including:
- `diagnostic` — structural/diagnostic finding
- `rhythm` — rhythm-related statement
- `diagnostic_class` / `diagnostic_subclass` — grouped diagnostic labels

Our four target classes (SR, AFIB, SBRAD, STACH) are **rhythm** statements (`rhythm == 1`), not diagnostic superclasses.

In [ ]:
scp = pd.read_csv(SCP_STATEMENTS_PATH, index_col=0)
scp.head()

In [ ]:
# All rhythm-related SCP codes in PTB-XL
rhythm_codes = scp[scp["rhythm"] == 1.0].copy()
print(f"Total rhythm SCP codes: {len(rhythm_codes)}")
display(rhythm_codes[["description", "SCP-ECG Statement Description"]])

print("\n--- Our 4 target rhythm codes ---")
for code in TARGET_RHYTHMS:
    row = scp.loc[code]
    print(f"  {code:6s}  rhythm={row['rhythm']}  {row['description']}")

In [ ]:
# Example scp_codes entries from the metadata
metadata["scp_codes_parsed"] = metadata["scp_codes"].apply(ast.literal_eval)

examples = metadata.sample(5, random_state=RANDOM_SEED)[["ecg_id", "patient_id", "report", "scp_codes"]]
display(examples)

## 5. Map Annotations to Target Rhythm Labels

**Labeling rule (conservative, no silent resolution):**

1. Parse `scp_codes` into a dictionary.
2. Check which of `{SR, AFIB, SBRAD, STACH}` appear as keys.
3. If **exactly one** target rhythm is present → assign it to `target_rhythm`.
4. If **zero** targets → exclude (`exclusion_reason = 'no_target_rhythm'`).
5. If **multiple** targets → exclude (`exclusion_reason = 'conflicting_rhythm_labels'`).

We do **not** pick the highest-confidence code or use list-order heuristics.

In [ ]:
def assign_target_rhythm(scp_dict: dict) -> tuple[str | None, str | None]:
    """Return (label, exclusion_reason). label is None when excluded."""
    present = [r for r in TARGET_RHYTHMS if r in scp_dict]
    if len(present) == 0:
        return None, "no_target_rhythm"
    if len(present) > 1:
        return None, f"conflicting_rhythm_labels:{','.join(present)}"
    return present[0], None


labels = metadata["scp_codes_parsed"].apply(assign_target_rhythm)
metadata["target_rhythm"] = labels.apply(lambda x: x[0])
metadata["exclusion_reason"] = labels.apply(lambda x: x[1])

# Show examples of conflicting labels (if any)
conflicts = metadata[metadata["exclusion_reason"].str.startswith("conflicting", na=False)]
print(f"Conflicting rhythm labels: {len(conflicts)}")
if len(conflicts) > 0:
    display(conflicts[["ecg_id", "scp_codes", "exclusion_reason"]].head(10))

## 6. Check Class Counts

In [ ]:
usable = metadata[metadata["target_rhythm"].notna()].copy()
excluded = metadata[metadata["target_rhythm"].isna()].copy()

print(f"Total recordings:     {len(metadata):,}")
print(f"Usable (4 classes):   {len(usable):,}")
print(f"Excluded:             {len(excluded):,}")
print("\nExclusion breakdown:")
print(excluded["exclusion_reason"].value_counts())

class_counts = usable["target_rhythm"].value_counts().reindex(TARGET_RHYTHMS)
class_pct = (class_counts / class_counts.sum() * 100).round(2)

count_table = pd.DataFrame({"count": class_counts, "pct": class_pct})
print("\n--- Class count table ---")
display(count_table)

## 7. Class Counts Summary

Class counts are computed above in Section 6. Full publication-style visualizations appear in **Exploratory Data Analysis and Visualization** (after patient-level split verification).

In [ ]:
# Quick sanity check: class counts sum to usable recordings (before QC removal)
assert class_counts.sum() == len(usable)
print(f"Class counts sum to usable recordings: {class_counts.sum():,} ✓")

## 8. Load Actual ECG Signals

We use the 100 Hz low-resolution records (`filename_lr`) via `wfdb.rdsamp`.
Each recording is a 12-lead, 10-second signal → shape `(1000, 12)` from wfdb, transposed to `(12, 1000)` for modeling.

In [ ]:
def load_ecg(filename_lr: str, data_dir: Path = RECORDS_DIR) -> tuple[np.ndarray, dict]:
    """Load one 12-lead ECG at 100 Hz. Returns signal (12, n_samples) and metadata."""
    signal, meta = wfdb.rdsamp(str(data_dir / filename_lr))
    return signal.T, meta  # (12, time_steps)


# Load one example to verify
sample_row = usable.iloc[0]
sample_signal, sample_meta = load_ecg(sample_row["filename_lr"])
print(f"Example file: {sample_row['filename_lr']}")
print(f"Signal shape: {sample_signal.shape}")
print(f"wfdb metadata keys: {list(sample_meta.keys())[:8]} ...")

## 9. Plot Representative ECGs for Each Class

Standard 12-lead layout: 3 rows × 4 columns. Lead II is commonly used for rhythm analysis.

In [ ]:
LEAD_NAMES = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
TIME_AXIS = np.arange(sample_signal.shape[1]) / SAMPLING_RATE  # seconds


def plot_12lead_ecg(signal: np.ndarray, title: str, fs: int = SAMPLING_RATE):
    """Plot all 12 leads in a standard 3×4 grid."""
    t = np.arange(signal.shape[1]) / fs
    fig, axes = plt.subplots(3, 4, figsize=(16, 8), sharex=True)
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i, ax in enumerate(axes.flat):
        ax.plot(t, signal[i], linewidth=0.6, color="black")
        ax.set_ylabel(LEAD_NAMES[i], fontsize=9, rotation=0, labelpad=20)
        ax.set_yticks([])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    axes[-1, 0].set_xlabel("Time (s)")
    axes[-1, 1].set_xlabel("Time (s)")
    axes[-1, 2].set_xlabel("Time (s)")
    axes[-1, 3].set_xlabel("Time (s)")
    plt.tight_layout()
    plt.show()


for rhythm in TARGET_RHYTHMS:
    row = usable[usable["target_rhythm"] == rhythm].iloc[0]
    sig, _ = load_ecg(row["filename_lr"])
    plot_12lead_ecg(sig, title=f"Example ECG — {rhythm}  (ecg_id={row['ecg_id']})")

## 10. Inspect Signal Dimensions, Sampling Rate, and Duration

In [ ]:
# Inspect a random subset of recordings for consistency
CHECK_N = min(200, len(usable))
check_rows = usable.sample(CHECK_N, random_state=RANDOM_SEED)

shapes, durations, fs_values = [], [], []
for _, row in check_rows.iterrows():
    sig, meta = load_ecg(row["filename_lr"])
    shapes.append(sig.shape)
    fs_values.append(meta["fs"])
    durations.append(sig.shape[1] / meta["fs"])

print(f"Checked {CHECK_N} recordings:")
print(f"  Unique shapes:     {set(shapes)}")
print(f"  Sampling rates:    {set(fs_values)}")
print(f"  Duration (s):      min={min(durations):.1f}, max={max(durations):.1f}, mean={np.mean(durations):.1f}")
print(f"  Leads:             {sig.shape[0]}")
print(f"  Samples per lead:  {sig.shape[1]}")

## 11. Check for Missing / Corrupted Data

We check:
- Whether signal files exist on disk
- NaN / Inf values in loaded signals
- Flat (zero-variance) leads
- Amplitude ranges

In [ ]:
def check_signal_file(row) -> dict:
    """Check one recording for file existence and signal quality issues."""
    filepath = RECORDS_DIR / row["filename_lr"]
    result = {"ecg_id": row["ecg_id"], "missing_file": not filepath.with_suffix("").exists()}
    if result["missing_file"]:
        result.update({"has_nan": None, "has_inf": None, "flat_leads": None, "amp_min": None, "amp_max": None})
        return result
    try:
        sig, _ = load_ecg(row["filename_lr"])
        result["has_nan"] = bool(np.isnan(sig).any())
        result["has_inf"] = bool(np.isinf(sig).any())
        result["flat_leads"] = int((sig.std(axis=1) == 0).sum())
        result["amp_min"] = float(sig.min())
        result["amp_max"] = float(sig.max())
    except Exception as e:
        result["missing_file"] = True
        result["error"] = str(e)
    return result


# Check all usable recordings (may take a few minutes)
quality_checks = pd.DataFrame([check_signal_file(row) for _, row in usable.iterrows()])

print(f"Missing / unreadable files: {quality_checks['missing_file'].sum()}")
print(f"Signals with NaN:           {quality_checks['has_nan'].sum()}")
print(f"Signals with Inf:           {quality_checks['has_inf'].sum()}")
print(f"Signals with flat leads:    {(quality_checks['flat_leads'] > 0).sum()}")
print(f"\nAmplitude range (mV) across all checked signals:")
print(f"  min: {quality_checks['amp_min'].min():.3f}")
print(f"  max: {quality_checks['amp_max'].max():.3f}")

# Mark corrupted recordings
bad_ids = quality_checks.loc[
    quality_checks["missing_file"] | quality_checks["has_nan"].fillna(False) | quality_checks["has_inf"].fillna(False),
    "ecg_id",
]
usable_clean = usable[~usable["ecg_id"].isin(bad_ids)].copy()
usable_clean["exclusion_reason"] = None  # reset for clean set

print(f"\nAfter removing corrupted: {len(usable_clean):,} usable recordings")

## 12. Decide and Document Preprocessing

**Decisions for Week 1 (documented, not yet aggressive):**

| Decision | Choice | Rationale |
|----------|--------|-----------|
| Sampling rate | 100 Hz (`filename_lr`) | Standard for PTB-XL benchmarks; 10 s × 100 = 1000 samples |
| Signal shape | `(12, 1000)` — leads × time | Channel-first format for downstream CNN/LSTM |
| Normalization | Per-channel Z-score per recording | Removes amplitude scale differences between leads/patients without global statistics leakage |
| Filtering | None (Week 1) | Inspect raw signals first; bandpass can be added in Week 2 if needed |
| Resampling | None | All 100 Hz records are 10 s / 1000 samples |
| Label scope | SR, AFIB, SBRAD, STACH only | AFLT excluded (extreme minority, ~46 records); SARRH/SVTAC excluded from main experiment |

In [ ]:
def z_score_normalize(signal: np.ndarray) -> np.ndarray:
    """Per-channel Z-score: (x - mean) / std along time axis."""
    mean = signal.mean(axis=1, keepdims=True)
    std = signal.std(axis=1, keepdims=True)
    std = np.where(std == 0, 1.0, std)  # avoid division by zero on flat leads
    return (signal - mean) / std


# Demonstrate on one signal
demo_sig, _ = load_ecg(usable_clean.iloc[0]["filename_lr"])
demo_norm = z_score_normalize(demo_sig)
print(f"Raw:  mean={demo_sig.mean():.4f}, std={demo_sig.std():.4f}")
print(f"Norm: mean={demo_norm.mean():.4f}, std={demo_norm.std():.4f} (per channel ≈ 0, 1)")

## 13. Create a Patient-Level Train / Validation / Test Split

**Critical:** Split by `patient_id`, NOT by individual ECG recordings.
A patient with multiple ECGs must appear in only one split.

Strategy:
1. Assign each patient a **primary rhythm label** (most common rhythm across their recordings).
2. Stratified split of patients: 70% train / 15% val / 15% test.
3. Assign all of a patient's recordings to the same split.

In [ ]:
# Step 1: patient-level primary label
patient_labels = (
    usable_clean.groupby("patient_id")["target_rhythm"]
    .agg(lambda x: x.value_counts().index[0])  # most common rhythm for this patient
    .reset_index()
    .rename(columns={"target_rhythm": "primary_rhythm"})
)

patients = patient_labels["patient_id"].values
patient_y = patient_labels["primary_rhythm"].values

# Step 2: stratified patient split (70 / 15 / 15)
p_train, p_temp, y_train, y_temp = train_test_split(
    patients, patient_y, test_size=0.30, stratify=patient_y, random_state=RANDOM_SEED
)
p_val, p_test, y_val, y_test = train_test_split(
    p_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_SEED
)

train_patients = set(p_train)
val_patients = set(p_val)
test_patients = set(p_test)

# Step 3: assign recordings to splits via patient membership
def assign_split(patient_id):
    if patient_id in train_patients:
        return "train"
    if patient_id in val_patients:
        return "val"
    return "test"

usable_clean["split"] = usable_clean["patient_id"].apply(assign_split)

print("Split sizes (recordings):")
print(usable_clean["split"].value_counts())
print("\nSplit sizes (patients):")
print(f"  train: {len(train_patients):,}")
print(f"  val:   {len(val_patients):,}")
print(f"  test:  {len(test_patients):,}")

## 14. Verify NO Patient Overlap Between Splits

In [ ]:
overlap_train_val = train_patients & val_patients
overlap_train_test = train_patients & test_patients
overlap_val_test = val_patients & test_patients

print("=== Patient Overlap Checks ===")
print(f"train ∩ val  = {len(overlap_train_val)} patients  {'✓ PASS' if len(overlap_train_val) == 0 else '✗ FAIL'}")
print(f"train ∩ test = {len(overlap_train_test)} patients  {'✓ PASS' if len(overlap_train_test) == 0 else '✗ FAIL'}")
print(f"val ∩ test   = {len(overlap_val_test)} patients  {'✓ PASS' if len(overlap_val_test) == 0 else '✗ FAIL'}")
print(f"\nTotal unique patients in splits: {len(train_patients) + len(val_patients) + len(test_patients):,}")
print(f"Expected (usable patients):       {usable_clean['patient_id'].nunique():,}")

---
# Exploratory Data Analysis and Visualization

The purpose of this section is to visually inspect class imbalance, representative ECG morphology, dataset split composition, and signal quality before downstream model development.

All figures use the cleaned dataset (`usable_clean`) and existing split/QC variables from above.

### Visualization 1 — Class Distribution (Counts)

This bar chart shows how many usable ECG recordings belong to each of the four target rhythm classes. It highlights the class imbalance that motivates investigating synthetic data augmentation in later weeks.

In [ ]:
eda_counts = usable_clean["target_rhythm"].value_counts().reindex(TARGET_RHYTHMS)
eda_pct = (eda_counts / eda_counts.sum() * 100).round(2)

eda_table = pd.DataFrame({
    "Class": TARGET_RHYTHMS,
    "Count": eda_counts.values,
    "Percentage": eda_pct.values,
})
print("Class | Count | Percentage")
display(eda_table)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(TARGET_RHYTHMS, eda_counts.values, edgecolor="black")
ax.set_title("PTB-XL Rhythm Class Distribution")
ax.set_xlabel("Rhythm Class")
ax.set_ylabel("Number of ECG Recordings")
for bar, count in zip(bars, eda_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(eda_counts.values) * 0.01,
        str(int(count)),
        ha="center",
        va="bottom",
        fontsize=10,
    )
plt.tight_layout()
plt.show()

assert eda_counts.sum() == len(usable_clean)

In [ ]:
majority_class = eda_counts.idxmax()
minority_class = eda_counts.idxmin()
print(
    f"The class distribution is substantially imbalanced, with {majority_class} representing "
    f"the largest class ({eda_pct[majority_class]:.1f}%) and {minority_class} the smallest "
    f"({eda_pct[minority_class]:.1f}%) among the four selected rhythms."
)
print(f"Total usable recordings after QC: {int(eda_counts.sum()):,}")

### Visualization 2 — Class Distribution (Percentages)

The same distribution expressed as a percentage of all usable recordings, making the relative rarity of minority classes easier to compare.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(TARGET_RHYTHMS, eda_pct.values, edgecolor="black")
ax.set_title("PTB-XL Rhythm Class Distribution (%)")
ax.set_xlabel("Rhythm Class")
ax.set_ylabel("Percentage of Recordings")
for bar, pct in zip(bars, eda_pct.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=10,
    )
plt.tight_layout()
plt.show()

### Visualization 3 — Representative ECG Signals

**Figure B (Lead II):** One representative recording per class, same lead and time axis, for direct waveform comparison.

**Figure A (12-lead):** Full 12-lead layout using the existing `plot_12lead_ecg()` function.

Selection is deterministic: the first valid recording in `usable_clean` for each class (after QC filtering).

In [ ]:
LEAD_IDX = 1  # Lead II — consistent across all classes

fig, axes = plt.subplots(len(TARGET_RHYTHMS), 1, figsize=(12, 2.5 * len(TARGET_RHYTHMS)), sharex=True)
for i, rhythm in enumerate(TARGET_RHYTHMS):
    row = usable_clean[usable_clean["target_rhythm"] == rhythm].iloc[0]
    sig, meta_i = load_ecg(row["filename_lr"])
    t = np.arange(sig.shape[1]) / meta_i["fs"]
    axes[i].plot(t, sig[LEAD_IDX], linewidth=0.8, color="black")
    axes[i].set_ylabel("Amplitude (mV)")
    axes[i].set_title(f"{rhythm} — {LEAD_NAMES[LEAD_IDX]} (ecg_id={row['ecg_id']})", loc="left")
    axes[i].spines["top"].set_visible(False)
    axes[i].spines["right"].set_visible(False)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Representative Lead II Traces by Rhythm Class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Figure A — 12-lead ECG per class (existing plotting function)
for rhythm in TARGET_RHYTHMS:
    row = usable_clean[usable_clean["target_rhythm"] == rhythm].iloc[0]
    sig, _ = load_ecg(row["filename_lr"])
    plot_12lead_ecg(sig, title=f"Example 12-Lead ECG — {rhythm} (ecg_id={row['ecg_id']})")

### Visualization 4 — Train / Validation / Test Distribution

Grouped bar chart showing that all four rhythm classes are represented in each patient-level split (train, validation, test). Uses the existing `split` column — the 70/15/15 patient-level logic is unchanged.

In [ ]:
split_order = ["train", "val", "test"]
split_counts = (
    usable_clean.groupby(["split", "target_rhythm"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=split_order, columns=TARGET_RHYTHMS)
)

display(split_counts)

fig, ax = plt.subplots(figsize=(9, 5))
split_counts.plot(kind="bar", ax=ax, edgecolor="black")
ax.set_title("Rhythm Distribution Across Patient-Level Splits")
ax.set_xlabel("Dataset Split")
ax.set_ylabel("Number of ECG Recordings")
ax.set_xticklabels(split_order, rotation=0)
ax.legend(title="Rhythm Class")
plt.tight_layout()
plt.show()

### Visualization 5 — Signal Quality

Distribution of amplitude range (peak-to-peak) and flat-lead counts from the existing `quality_checks` dataframe. Purpose: identify obvious signal-quality problems or unusual recordings before model development.

In [ ]:
qc_ok = quality_checks[~quality_checks["missing_file"]].copy()
qc_ok["amp_range"] = qc_ok["amp_max"] - qc_ok["amp_min"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(qc_ok["amp_range"].dropna(), bins=50, edgecolor="black")
axes[0].set_title("Signal Amplitude Range (Peak-to-Peak)")
axes[0].set_xlabel("Amplitude range (mV)")
axes[0].set_ylabel("Number of recordings")

axes[1].hist(qc_ok["flat_leads"].dropna(), bins=range(0, int(qc_ok["flat_leads"].max()) + 2), edgecolor="black")
axes[1].set_title("Flat Leads per Recording")
axes[1].set_xlabel("Count of flat leads")
axes[1].set_ylabel("Number of recordings")

plt.tight_layout()
plt.show()

print("QC summary (from existing quality_checks):")
print(f"  Missing / unreadable files: {quality_checks['missing_file'].sum()}")
print(f"  Signals with NaN:           {quality_checks['has_nan'].sum()}")
print(f"  Signals with Inf:           {quality_checks['has_inf'].sum()}")
print(f"  Signals with flat leads:    {(quality_checks['flat_leads'] > 0).sum()}")
print(f"  Amplitude range (mV):       {qc_ok['amp_min'].min():.3f} to {qc_ok['amp_max'].max():.3f}")

## 15. Simple Baseline Classifier

Purpose: verify the full pipeline `data → preprocessing → labels → train → val → test` works.

We use a **Random Forest** on simple hand-crafted features (mean, std, min, max per lead = 48 features).
No hyperparameter tuning. This is a sanity-check baseline, not a final model.

In [ ]:
def extract_features(signal: np.ndarray) -> np.ndarray:
    """48 simple features: mean, std, min, max for each of 12 leads."""
    return np.concatenate([
        signal.mean(axis=1),
        signal.std(axis=1),
        signal.min(axis=1),
        signal.max(axis=1),
    ])


def build_feature_matrix(df: pd.DataFrame, max_samples: int = 2000) -> tuple[np.ndarray, np.ndarray]:
    """Load signals and extract features. Subsample for speed in Week 1."""
    subset = df if len(df) <= max_samples else df.sample(max_samples, random_state=RANDOM_SEED)
    X, y = [], []
    for _, row in subset.iterrows():
        sig, _ = load_ecg(row["filename_lr"])
        sig = z_score_normalize(sig)
        X.append(extract_features(sig))
        y.append(row["target_rhythm"])
    return np.array(X), np.array(y)


print("Building feature matrices (subsampled for Week 1 speed)...")
train_df = usable_clean[usable_clean["split"] == "train"]
val_df = usable_clean[usable_clean["split"] == "val"]
test_df = usable_clean[usable_clean["split"] == "test"]

X_train, y_train = build_feature_matrix(train_df, max_samples=2000)
X_val, y_val = build_feature_matrix(val_df, max_samples=500)
X_test, y_test = build_feature_matrix(test_df, max_samples=500)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)
clf.fit(X_train_s, y_train)

print(f"\nTrain accuracy: {clf.score(X_train_s, y_train):.3f}")
print(f"Val accuracy:   {clf.score(X_val_s, y_val):.3f}")
print(f"Test accuracy:  {clf.score(X_test_s, y_test):.3f}")

print("\n--- Test Set Classification Report ---")
y_pred = clf.predict(X_test_s)
print(classification_report(y_test, y_pred, labels=TARGET_RHYTHMS))

print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred, labels=TARGET_RHYTHMS))

## 16. Record Observations and Research Questions

### Key Observations

- PTB-XL rhythm labels live in `scp_codes` as SCP statement keys, not in a separate `target_rhythm` column.
- SR dominates the dataset; STACH and SBRAD are moderate minorities; AFIB is the largest minority among the four.
- Multiple rhythm codes can co-occur in one record — we exclude these rather than guessing.
- All 100 Hz records are 12-lead, 10-second, 1000-sample signals.
- Patient-level splitting is essential because many patients have multiple ECG recordings.
- The simple Random Forest baseline confirms the pipeline works end-to-end.

---
## Week 1 Findings

*(Facts below are generated from this notebook's outputs — re-run all cells to refresh.)*

In [ ]:
train_acc = clf.score(X_train_s, y_train)
val_acc = clf.score(X_val_s, y_val)
test_acc = clf.score(X_test_s, y_test)

findings = {
    "total_recordings": len(metadata),
    "total_patients": metadata["patient_id"].nunique(),
    "usable_recordings": len(usable_clean),
    "excluded_recordings": len(metadata) - len(usable_clean),
    "class_counts": eda_counts.to_dict(),
    "class_percentages": eda_pct.to_dict(),
    "class_counts_sum_check": int(eda_counts.sum()),
    "signal_shape": "(12, 1000)",
    "sampling_rate_hz": SAMPLING_RATE,
    "duration_seconds": 10,
    "train_recordings": int((usable_clean["split"] == "train").sum()),
    "val_recordings": int((usable_clean["split"] == "val").sum()),
    "test_recordings": int((usable_clean["split"] == "test").sum()),
    "train_patients": len(train_patients),
    "val_patients": len(val_patients),
    "test_patients": len(test_patients),
    "patient_overlap_train_val": len(overlap_train_val),
    "patient_overlap_train_test": len(overlap_train_test),
    "patient_overlap_val_test": len(overlap_val_test),
    "conflicting_labels_excluded": int(metadata["exclusion_reason"].str.startswith("conflicting", na=False).sum()),
    "no_target_rhythm_excluded": int((metadata["exclusion_reason"] == "no_target_rhythm").sum()),
    "corrupted_signals_excluded": len(bad_ids),
    "qc_missing_files": int(quality_checks["missing_file"].sum()),
    "qc_nan_signals": int(quality_checks["has_nan"].sum()),
    "qc_inf_signals": int(quality_checks["has_inf"].sum()),
    "qc_flat_lead_signals": int((quality_checks["flat_leads"] > 0).sum()),
    "baseline_train_accuracy": round(train_acc, 3),
    "baseline_val_accuracy": round(val_acc, 3),
    "baseline_test_accuracy": round(test_acc, 3),
}

print("=" * 55)
print("WEEK 1 FINDINGS")
print("=" * 55)
for key, val in findings.items():
    print(f"  {key}: {val}")

# Verification checks
assert eda_counts.sum() == len(usable_clean)
assert len(overlap_train_val) == 0 and len(overlap_train_test) == 0 and len(overlap_val_test) == 0
print("\nVerification:")
print(f"  SR+AFIB+STACH+SBRAD = {int(eda_counts.sum()):,} usable recordings ✓")
print(f"  Patient overlap across splits = 0 ✓")

---
## Research Decisions / Open Questions

Questions to address in **Week 2** and beyond:

1. **Which synthetic-data generation method should we use?**
   - Reuse an existing PTB-XL-compatible generator from the reference paper, or train our own?
   - Must generate 12-lead, 10 s, 100 Hz signals to match our preprocessing pipeline.

2. **How should synthetic ECG quality be measured?**
   - Candidate measures: statistical similarity, temporal similarity (DTW), feature-space distance.
   - Need to define thresholds for "high quality" vs "low quality" synthetic samples.

3. **Should quality be assessed in raw signal space, feature space, or both?**
   - Raw signal: preserves morphology but is high-dimensional.
   - Feature space: faster but may miss clinically relevant waveform details.

4. **What synthetic-to-real ratio should be tested?**
   - Given SR >> AFIB/SBRAD/STACH imbalance, minority classes likely benefit most from augmentation.
   - Need ablation: 1×, 2×, 5× synthetic-to-real ratios per minority class.

5. **What baseline models should be compared against CNN-LSTM?**
   - Week 1 Random Forest (hand-crafted features) is a lower bound.
   - Need: real-only CNN-LSTM, random-augmentation CNN-LSTM, quality-aware-augmentation CNN-LSTM.
   - Also compare against training on all synthetic data (no quality filter).

---

**Week 1 status:** Dataset understood, labels constructed, EDA complete, preprocessing documented, patient-level split verified, baseline pipeline confirmed. Ready for Week 2.